## Cityscapes-trained EoMT - BENCHMARK

## retrieve predictions and compute mIoU

We do not need to run inference on the EoMT Cityscapes-trained model to get predictions. We already have the predictions, so we only compute the mIoU metric over all 19 class labels.

In [1]:
import yaml
from lightning import seed_everything
import torch
import numpy as np
import warnings
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image


warnings.filterwarnings("ignore")

seed_everything(0, verbose=False)

device = 0

/home/elisa/outlierdrive/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import glob

pred_city = "../data/eomt_valset_predictions/cityscapes_model/png/content/drive/MyDrive/eomt_valset_predictions/cityscapes_model/png"

gt_paths = sorted(glob.glob(
    "../data/datasets_unzip/gtFine_trainIds/val/*/*_gtFine_labelTrainIds.png"
))

In [11]:
NUM_CLASSES = 19
IGNORE_INDEX = 255

def load_mask(path):
    return np.array(Image.open(path), dtype=np.int64)

def confusion_matrix(gt, pred, num_classes=19):

    gt = gt.astype(np.int64)
    pred = pred.astype(np.int64)
    
    valid_mask = (
        (gt != IGNORE_INDEX) &
        (gt >= 0) & (gt < num_classes) &
        (pred >= 0) & (pred < num_classes)
    )

    hist = np.bincount(
        num_classes * gt[valid_mask] + pred[valid_mask],
        minlength=num_classes ** 2
    ).reshape(num_classes, num_classes)

    return hist

In [15]:
def eval_pred_all_cityscapes(gt_paths, pred_dir):
    total_hist = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float64)
    missing_predictions = []

    for gt_path in tqdm(gt_paths):
        gt_filename = os.path.basename(gt_path)

        pred_filename = gt_filename.replace(
            "_gtFine_labelTrainIds.png",
            "_predTrainIds.png"
        )

        pred_path = os.path.join(pred_dir, pred_filename)

        if not os.path.exists(pred_path):
            missing_predictions.append(pred_path)
            continue

        gt = load_mask(gt_path)
        pred = load_mask(pred_path)

        if gt.shape != pred.shape:
            raise ValueError(f"Shape mismatch: {gt.shape} vs {pred.shape}")

        total_hist += confusion_matrix(gt, pred, NUM_CLASSES)

    intersection = np.diag(total_hist)
    union = total_hist.sum(axis=1) + total_hist.sum(axis=0) - intersection

    class_iou = intersection / np.maximum(union, 1)
    mean_iou = np.mean(class_iou)

    pixel_accuracy = intersection.sum() / np.maximum(total_hist.sum(), 1)

    return {
        "class_iou": class_iou,
        "mean_iou": mean_iou,
        "pixel_accuracy": pixel_accuracy,
        "confusion_matrix": total_hist,
        "missing_predictions": missing_predictions,
    }

In [13]:
results_city = eval_pred_all_cityscapes(gt_paths, pred_city)

100%|██████████| 500/500 [00:23<00:00, 21.19it/s]


In [14]:
import pandas as pd

summary_df = pd.DataFrame({
    "Model": [
        "Cityscapes-trained EoMT (all 19 classes)"
    ],
    "mIoU (%)": [
        results_city["mean_iou"] * 100
    ],
    "Pixel Accuracy (%)": [
        results_city["pixel_accuracy"] * 100
    ]
})

summary_df

,Model,mIoU (%),Pixel Accuracy (%)
0,Cityscapes-trained EoMT (all 19 classes),81.679691,96.718217


In [18]:
CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle"
]


per_class_iou_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Cityscapes-trained IoU (%)": results_city["class_iou"] * 100
})

per_class_iou_df

,Class,Cityscapes-trained IoU (%)
0,road,98.396221
1,sidewalk,87.359580
2,building,94.149436
3,wall,66.065447
4,fence,65.494912
5,pole,71.037070
6,traffic light,75.000194
7,traffic sign,82.126556
8,vegetation,93.016663
9,terrain,66.598365


Notice that mIoI value is a bit lower than the evaluation on the overlap classes. This is because some classes are actually difficult, like rider, pole, terrain. Recall, that they were not present in the overlap class labels space. So their IoU may be mush lower, which correspondingly lowers the average.

The good thing is that when we run the evaluation using the command provided in th eEoMT repository, we got:

- EoMT Cityscapes val mIoU = 81.6797

which is essentually the same. It confirms one more time that our manual evaluation pipeline is reproducing the repository's evaluation.

---
## Fine-tuning COCO-trained EoMT on Cityscapes dataset

Step 1. Start with the head-only fine-tuning.